In previous posts I looked at how to discretize the Stokes equations, which are appropriate for low Reynolds-number fluid flow.
Here I'll look at how we can discretize the full Navier-Stokes equations:
$$\frac{\partial}{\partial t}\rho u + \nabla\cdot\rho u\otimes u = -\nabla p + \nabla\cdot \tau$$
where the deviatoric stress tensor is $\tau = 2\mu\dot\varepsilon$.
I'll use a projection method, which proceeds in two steps: (1) step the velocity forward in time, ignoring the pressure gradient; (2) compute the pressure field by solving the potential equation; and (3) use the newly-computed pressure field to project the candidate velocity from the first step back into the space of divergence-free velocity fields.

In [ ]:
import gmsh
gmsh.initialize()

In [ ]:
import numpy as np
from numpy import pi as π

Lx = 6.0
Ly = 2.0
lcar = 1 / 16

gmsh.model.add("chamber")
geo = gmsh.model.geo
ps = [(0, 0), (Lx, 0), (Lx, Ly), (0, Ly)]
box_points = [geo.add_point(*p, 0) for p in ps]
box_lines = [
    geo.add_line(i1, i2) for i1, i2 in zip(box_points, np.roll(box_points, 1))
]

for line in box_lines:
    geo.add_physical_group(1, [line])

c = np.array([Lx / 2, Ly / 2, 0])
#center = geo.add_point(*c)
r = Ly / 8
num_circle_points = 16
θs = np.linspace(0.0, 2 * π, num_circle_points + 1)[:-1]
ss = np.column_stack((np.cos(θs), np.sin(θs), np.zeros(num_circle_points)))
tie_points = [geo.add_point(*(c + r * s)) for s in ss]
circle_arcs = [
    geo.add_line(p1, p2)
    #geo.add_circle_arc(p1, center, p2)
    for p1, p2 in zip(tie_points, np.roll(tie_points, 1))
]

geo.add_physical_group(1, circle_arcs)

outer_curve_loop = geo.add_curve_loop(box_lines)
inner_curve_loop = geo.add_curve_loop(circle_arcs)
plane_surface = geo.add_plane_surface([outer_curve_loop, inner_curve_loop])
geo.add_physical_group(2, [plane_surface])
geo.synchronize()
gmsh.model.mesh.generate(2)
gmsh.write("chamber.msh")

In [ ]:
import firedrake
mesh = firedrake.Mesh("chamber.msh")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.set_aspect("equal")
firedrake.triplot(mesh, axes=ax)
ax.legend(loc="upper right");